# CyberAccountant pipeline — interactive driver

Run cells top to bottom. Each section runs one stage and shows its artifact.

Config: change `ID_STYLE` between `"mnemonic"` and `"numeric"` to compare.

In [ ]:
from pathlib import Path
from cyberaccountant import Pipeline, PipelineConfig

ID_STYLE = "mnemonic"          # or "numeric"
TAG_MODEL = "deepseek/deepseek-v4.1-flash-20261017"
REL_MODEL = "~deepseek/deepseek-pro-latest"

cfg = PipelineConfig(
    data_dir=Path("../data_md"),
    out_dir=Path("../out"),
    id_style=ID_STYLE,
    tag_model=TAG_MODEL,
    rel_model=REL_MODEL,
)
p = Pipeline(cfg)

## Stage 0 — ingest / rename / dedup

In [ ]:
mapping = p.ingest()
import pandas as pd
pd.DataFrame(mapping).T

## Stage 1 — entity tagging (LLM, per file)

In [ ]:
tagged = p.tag()
list(tagged.keys())[:5], len(tagged)

## Stage 2 — global entity registry

In [ ]:
reg = p.registry()
import pandas as pd
rows = [{"num": e.num, "mnemonic": e.mnemonic, "type": e.type,
         "raw": e.raw, "value": e.value, "unit": e.unit, "file": e.file,
         "role": e.role} for e in reg.entities]
df = pd.DataFrame(rows)
df.head(20)

In [ ]:
df["type"].value_counts()

## Stage 3 — replace tags with ids

In [ ]:
replaced = p.replace()
h8 = next(iter(replaced))
print(replaced[h8][:1200])

## Stage 4 — scope checks (contract / period)

In [ ]:
scope = p.scope()
scope

## Stage 5 — relation extraction (LLM, whole corpus)

In [ ]:
relations_raw = p.relations()
len(relations_raw), relations_raw[:5]

## Stage 6 — solve the system

In [ ]:
rels, fishy, verdicts, dangling = p.solve()
import pandas as pd
rows = [{"i": r.index, "equation": r.equation, "tag": r.tag,
         "status": r.status, "error": r.error} for r in rels]
pd.DataFrame(rows)

## Stage 7 — report

In [ ]:
report = p.report()
import json
print(json.dumps(report["summary"], indent=2, ensure_ascii=False))

In [ ]:
import pandas as pd
pd.DataFrame(report["card"]["cells"])

In [ ]:
print("duplicates:", report["files"]["duplicates"])
print("out_of_scope:", report["files"]["out_of_scope"])
print("dangling:", report["files"]["dangling"])
print("fishy:", report["fishy"])